In [ ]:
import os

# --- 1. THIẾT LẬP HẰNG SỐ SEED ---
SEED = 42

# --- 2. CẤU HÌNH BIẾN MÔI TRƯỜNG ---

# PYTHONHASHSEED:
# - Python dùng hàm băm (hash) ngẫu nhiên cho Dictionary/Set để bảo mật.
# - Cần cố định nó để đảm bảo việc xử lý chuỗi (Text) luôn ra kết quả giống nhau.
os.environ['PYTHONHASHSEED'] = str(SEED)

# --- 3. IMPORT THƯ VIỆN ---
import random
import numpy as np
import pandas as pd
import tensorflow as tf

# --- 4. HÀM THIẾT LẬP SEED TOÀN CỤC ---
def set_seed(seed=SEED):
    """
    Hàm này khóa (freeze) tất cả các bộ sinh số ngẫu nhiên.
    Đảm bảo tính tái lập (Reproducibility) của thí nghiệm.
    """
    # Khóa seed của thư viện chuẩn Python (dùng cho random.choice, shuffle...)
    random.seed(seed)
    
    # Khóa seed của Numpy (dùng cho np.random, pandas sample, scikit-learn split...)
    np.random.seed(seed)
    
    # Khóa seed của TensorFlow (dùng cho việc khởi tạo trọng số Weight/Bias ban đầu)
    tf.random.set_seed(seed)
    
    print(f"Thiết lập Global Seed: {seed}")

# Gọi hàm ngay lập tức để áp dụng cấu hình
set_seed(SEED)

# --- 5. KIỂM TRA DỮ LIỆU ĐẦU VÀO ---
print("\nDanh sách file dữ liệu tìm thấy:")
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# Chọn tập data Phishing-site-urls làm tập train và tập Real-world-phishing làm tập test
path_data1 = ""
path_data2 = ""

# Lệnh này quét toàn bộ thư mục Input
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        full_path = os.path.join(dirname, filename)

        if "phishing_site_urls.csv" in filename:
            path_data1 = full_path
            print(f"Tập data1 tại: {path_data1}")

        elif "phishing_url_dataset_unique.csv" in filename:
            path_data2 = full_path
            print(f"Tập data2 tại: {path_data2}")

In [ ]:
import re

# Tạm thời xóa giao thức, chỉ giữ lại domain url
def clean_url(url):
    # 1. Chuyển thành chuỗi và chữ thường -> giúp giảm độ phức tạp mô hình -> giảm overfiting
    url = str(url).lower()
    
    # 2. Xóa protocol và www
    url = re.sub(r'^https?://', '', url)
    url = re.sub(r'^www\.', '', url)
    
    return url

In [ ]:
# Xử lý bộ dữ liệu

# 1. Đọc file
df_data1 = pd.read_csv(path_data1)
df_data2 = pd.read_csv(path_data2)

In [ ]:
df_data1.head()

In [ ]:
df_data2.drop(columns='source', inplace=True) # không cần cột này nên loại bỏ
df_data2.head()

In [ ]:
# 2. Chuyển nhãn từ chữ sang số
# Nếu là 'bad' thì thành 1, ngược lại là 0
df_data1['Label'] = df_data1['Label'].apply(lambda x: 1 if x == 'bad' else 0)

# 3. Clean URL
df_data1['URL'] = df_data1['URL'].apply(lambda x: clean_url(x))

# Xem thử 5 dòng đầu tiên
print(df_data1.head())

In [ ]:
# Kiểm tra lại file data1
# Kiểm tra giá trị rỗng
df_data1.isnull().sum().sum()

In [ ]:
df_data1['Label'].value_counts()
# 1 là phishing
# 0 là an toàn

In [ ]:
# Xử lý bộ dữ liệu data2
df_data2.rename(columns={'url': 'URL'}, inplace=True)
df_data2.rename(columns={'label': 'Label'}, inplace=True)

In [ ]:
# Clean URL
df_data2['URL'] = df_data2['URL'].apply(lambda x: clean_url(x))

# Xem thử 5 dòng đầu tiên
print(df_data2.head())

In [ ]:
# Kiểm tra lại file test
# Kiểm tra giá trị rỗng
df_data2.isnull().sum().sum()

In [ ]:
df_data2['Label'].value_counts()
# 1 là phishing
# 0 là an toàn

In [ ]:
# Gộp 2 dữ liệu và trộn
df_train = pd.concat([df_data1, df_data2], ignore_index=True)

# Xóa các dòng bị trùng lặp sau khi gộp tránh data leakage
df_train = df_train.drop_duplicates()
df_train = df_train.sample(frac=1, random_state=SEED).reset_index(drop=True)

In [ ]:
df_train['Label'].value_counts()
# 1 là phishing
# 0 là an toàn

In [ ]:
df_test = pd.read_csv('/kaggle/input/datasets/quangnguynv/real-dataset-30-phishing/phishing_dataset_full_46k.csv')
df_test['URL'] = df_test['URL'].apply(lambda x: clean_url(x))

In [ ]:
df_test.head()

In [ ]:
df_test['Label'].value_counts()
# 1 là phishing
# 0 là an toàn

In [ ]:
# Kiểm tra trùng lặp giữa train và test
overlap = set(df_train['URL']).intersection(set(df_test['URL']))
print(f"Số URL trùng giữa train và test: {len(overlap)}")

# Loại bỏ URL trùng khỏi tập train 
df_train = df_train[~df_train['URL'].isin(overlap)]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. BIỂU ĐỒ TRÒN: TỈ LỆ NHÃN 
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
df_train['Label'].value_counts().plot.pie(autopct='%1.1f%%', colors=['#66b3ff','#ff9999'], explode=(0.05, 0))
plt.title('Tỉ lệ Legit (0) vs Phishing (1)')
plt.ylabel('')

# 2. BIỂU ĐỒ CỘT: PHÂN BỐ ĐỘ DÀI URL
# Tính độ dài từng URL
url_lengths = df_train['URL'].apply(len)

plt.subplot(1, 2, 2)
sns.histplot(url_lengths, bins=50, kde=True, color='purple')
plt.axvline(x=500, color='r', linestyle='--', label='URL_LEN = 500') # Đường kẻ đỏ mốc cắt
plt.title('Phân bố độ dài URL')
plt.xlabel('Số ký tự')
plt.ylabel('Số lượng URL')
plt.legend()

plt.tight_layout()
plt.show()

# Thống kê nhanh 
print(f"   - Độ dài trung bình: {url_lengths.mean():.2f}")
print(f"   - URL dài nhất: {url_lengths.max()}")
print(f"   - Tỉ lệ URL ngắn hơn 500 ký tự: {(url_lengths < 500).mean()*100:.2f}%")

In [ ]:
# Trích xuất đặc trưng từ URL gốc
from collections import Counter
from urllib.parse import urlparse

def extract_basic_features(url):
    """Basic URL features (13 features)"""
    features = {}
    
    features['url_length'] = len(url)
    features['num_dots'] = url.count('.')
    features['num_hyphens'] = url.count('-')
    features['num_underscores'] = url.count('_')
    features['num_slashes'] = url.count('/')
    features['num_questionmarks'] = url.count('?')
    features['num_equals'] = url.count('=')
    features['num_at'] = url.count('@')
    features['num_ampersand'] = url.count('&')
    features['num_exclamation'] = url.count('!')
    features['num_tilde'] = url.count('~')
    features['num_percent'] = url.count('%')
    features['num_hash'] = url.count('#')
    
    return features

def extract_domain_features(url):
    """Domain and path features (8 features)"""
    features = {}
    
    try:
        parsed = urlparse('http://' + url)
        domain = parsed.netloc
        path = parsed.path
        query = parsed.query
        
        features['domain_length'] = len(domain)
        features['path_length'] = len(path)
        features['query_length'] = len(query)
        
        # Subdomains
        domain_parts = domain.split('.')
        features['num_subdomains'] = len(domain_parts) - 2 if len(domain_parts) > 2 else 0
        
        # TLD
        features['tld_length'] = len(domain_parts[-1]) if domain_parts else 0
        
        # Path depth
        path_parts = [p for p in path.split('/') if p]
        features['path_depth'] = len(path_parts)
        
        # Query parameters
        query_params = [p for p in query.split('&') if p]
        features['num_query_params'] = len(query_params)
        
        # Port
        features['has_port'] = 1 if ':' in domain and not domain.startswith('[') else 0
        
    except:
        features['domain_length'] = 0
        features['path_length'] = len(url)
        features['query_length'] = 0
        features['num_subdomains'] = 0
        features['tld_length'] = 0
        features['path_depth'] = 0
        features['num_query_params'] = 0
        features['has_port'] = 0
    
    return features

def extract_character_features(url):
    """Character-based features (8 features)"""
    features = {}
    
    features['num_digits'] = sum(c.isdigit() for c in url)
    features['num_letters'] = sum(c.isalpha() for c in url)
    features['digit_ratio'] = features['num_digits'] / len(url) if len(url) > 0 else 0
    features['letter_ratio'] = features['num_letters'] / len(url) if len(url) > 0 else 0
    
    # Vowels and consonants
    vowels = sum(1 for c in url.lower() if c in 'aeiou')
    consonants = sum(1 for c in url.lower() if c.isalpha() and c not in 'aeiou')
    features['vowel_ratio'] = vowels / len(url) if len(url) > 0 else 0
    features['consonant_ratio'] = consonants / len(url) if len(url) > 0 else 0
    
    # Uppercase ratio
    features['uppercase_ratio'] = sum(1 for c in url if c.isupper()) / len(url) if len(url) > 0 else 0
    
    # Special characters
    special_chars = sum(1 for c in url if not c.isalnum() and c not in '.-/:?=&#@!')
    features['special_char_ratio'] = special_chars / len(url) if len(url) > 0 else 0
    
    return features

def extract_suspicious_features(url):
    """Suspicious pattern features (10 features)"""
    features = {}
    
    # Suspicious keywords
    suspicious_words = [
        'login', 'verify', 'account', 'update', 'secure', 'signin',
        'banking', 'confirm', 'password', 'suspend', 'paypal', 'ebay',
        'amazon', 'admin', 'user', 'click', 'free', 'bonus'
    ]
    features['num_suspicious_words'] = sum(1 for word in suspicious_words if word in url.lower())
    
    # IP address
    features['has_ip'] = 1 if re.search(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', url) else 0
    
    # URL shorteners
    shorteners = ['bit.ly', 'goo.gl', 't.co', 'tinyurl', 'ow.ly', 'is.gd', 'buff.ly']
    features['is_shortened'] = 1 if any(s in url for s in shorteners) else 0
    
    # Suspicious TLDs
    try:
        parsed = urlparse('http://' + url)
        domain_parts = parsed.netloc.split('.')
        tld = domain_parts[-1] if domain_parts else ''
        
        suspicious_tlds = ['tk', 'ml', 'ga', 'cf', 'gq', 'xyz', 'top', 'work', 'click', 'link']
        features['tld_is_suspicious'] = 1 if tld in suspicious_tlds else 0
        
        # TLD type
        features['tld_is_country'] = 1 if len(tld) == 2 else 0
        features['tld_is_generic'] = 1 if tld in ['com', 'org', 'net', 'edu', 'gov'] else 0
    except:
        features['tld_is_suspicious'] = 0
        features['tld_is_country'] = 0
        features['tld_is_generic'] = 0
    
    # Double slashes
    features['has_double_slash'] = 1 if '//' in url[8:] else 0  # Skip protocol
    
    # @ symbol (credential hiding)
    features['has_at_symbol'] = 1 if '@' in url else 0
    
    # Multiple dots in domain (subdomain abuse)
    try:
        domain = urlparse('http://' + url).netloc
        features['excessive_dots'] = 1 if domain.count('.') > 3 else 0
    except:
        features['excessive_dots'] = 0
    
    # Hex encoding (obfuscation)
    features['has_hex_encoding'] = 1 if re.search(r'%[0-9a-fA-F]{2}', url) else 0
    
    return features

def extract_statistical_features(url):
    """Statistical features (8 features)"""
    features = {}
    
    # Entropy
    if len(url) > 0:
        freq = Counter(url)
        entropy = -sum((count/len(url)) * np.log2(count/len(url)) 
                      for count in freq.values())
        features['entropy'] = entropy
    else:
        features['entropy'] = 0
    
    # Character frequency stats
    if len(url) > 0:
        char_freq = Counter(url)
        frequencies = list(char_freq.values())
        
        features['max_char_frequency'] = max(frequencies)
        features['char_freq_std'] = np.std(frequencies)
        features['char_freq_mean'] = np.mean(frequencies)
    else:
        features['max_char_frequency'] = 0
        features['char_freq_std'] = 0
        features['char_freq_mean'] = 0
    
    # Consecutive characters
    max_consecutive = 1
    current_consecutive = 1
    for i in range(1, len(url)):
        if url[i] == url[i-1]:
            current_consecutive += 1
            max_consecutive = max(max_consecutive, current_consecutive)
        else:
            current_consecutive = 1
    features['max_consecutive_chars'] = max_consecutive
    
    # Unique character ratio
    features['unique_char_ratio'] = len(set(url)) / len(url) if len(url) > 0 else 0
    
    # Longest token length
    tokens = re.split(r'[/\.?=&-]', url)
    features['max_token_length'] = max(len(t) for t in tokens) if tokens else 0
    features['avg_token_length'] = np.mean([len(t) for t in tokens if t]) if tokens else 0
    
    return features

def extract_ngram_features(url):
    """N-gram pattern features (5 features)"""
    features = {}
    
    # Suspicious bigrams
    suspicious_bigrams = ['..', '//', '@@', '--', '__']
    for i, bigram in enumerate(suspicious_bigrams):
        features[f'has_suspicious_bigram_{i}'] = 1 if bigram in url else 0
    
    return features

def extract_features(url):
    """
    Extract ALL features from URL
    
    Total: 52 features
    - Basic: 13
    - Domain: 8
    - Character: 8
    - Suspicious: 10
    - Statistical: 8
    - N-gram: 5
    
    Returns:
        dict: Complete feature dictionary
    """
    features = {}
    
    # Combine all feature types
    features.update(extract_basic_features(url))
    features.update(extract_domain_features(url))
    features.update(extract_character_features(url))
    features.update(extract_suspicious_features(url))
    features.update(extract_statistical_features(url))
    features.update(extract_ngram_features(url))
    
    return features

# Extract features
features_list = []
labels_list = []

for idx, row in df_train.iterrows():
    if idx % 10000 == 0:
        print(f"  Progress: {idx:,}/{len(df_train):,} ({idx/len(df_train)*100:.1f}%)")
    
    features = extract_features(row['URL'])
    features_list.append(features)
    labels_list.append(row['Label'])

# Create feature DataFrame
X = pd.DataFrame(features_list)
y = np.array(labels_list)

print(f"\nFeature extraction complete!")
print(f"   Features: {X.shape[1]}")
print(f"   Samples:  {X.shape[0]:,}")
print(f"\nFeature list:")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2d}. {col}")

In [ ]:
from sklearn.model_selection import train_test_split

# Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Hyperparameters
N_ESTIMATORS = 300
MAX_DEPTH = 35
MIN_SAMPLES_SPLIT = 8
MIN_SAMPLES_LEAF = 4
MAX_FEATURES = 'sqrt'

print(f"\nHyperparameters:")
print(f"  n_estimators:      {N_ESTIMATORS}")
print(f"  max_depth:         {MAX_DEPTH}")
print(f"  min_samples_split: {MIN_SAMPLES_SPLIT}")
print(f"  min_samples_leaf:  {MIN_SAMPLES_LEAF}")
print(f"  max_features:      {MAX_FEATURES}")

# Create and train model
rf = RandomForestClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    min_samples_split=MIN_SAMPLES_SPLIT,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    max_features=MAX_FEATURES,
    class_weight='balanced',
    random_state=SEED,
    n_jobs=-1,
    verbose=1
)

rf.fit(X_train, y_train)

# Feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\n Top 15 Important Features:")
print(feature_importance.head(15).to_string(index=False))

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, roc_curve, accuracy_score
)
import numpy as np

# 1. Predict probabilities trên tập Validation
y_val_proba = rf.predict_proba(X_val)[:, 1]

# 2. Tính Precision-Recall curve
precisions, recalls, thresholds = precision_recall_curve(y_val, y_val_proba)

# 3. Tìm ngưỡng để Recall >= 95%
TARGET_RECALL = 0.95
recall_mask = recalls >= TARGET_RECALL

if recall_mask.any():
    # Lấy tất cả các index thỏa mãn Recall >= 95%
    valid_indices = np.where(recall_mask)[0]
    
    # Trong các index đó, chọn ngưỡng có Precision cao nhất (để giảm báo nhầm tối đa)
    best_idx = valid_indices[np.argmax(precisions[valid_indices])]
    
    # Xử lý biên (phòng trường hợp index vượt quá độ dài threshold)
    if best_idx < len(thresholds):
        OPTIMAL_THRESHOLD = thresholds[best_idx]
    else:
        OPTIMAL_THRESHOLD = 0.5 # Fallback
        
    print(f"\nOPTIMAL THRESHOLD (Recall >= {TARGET_RECALL*100:.0f}%): {OPTIMAL_THRESHOLD:.4f}")
    print(f"   - Recall:    {recalls[best_idx]:.4f} ({recalls[best_idx]*100:.1f}%)")
    print(f"   - Precision: {precisions[best_idx]:.4f} ({precisions[best_idx]*100:.1f}%)")

In [ ]:
test_features_list = []
for idx, row in df_test.iterrows():
    if idx % 10000 == 0:
        print(f"  Test Progress: {idx:,}/{len(df_test):,} ({idx/len(df_test)*100:.1f}%)")
    features = extract_features(row['URL'])
    test_features_list.append(features)

X_test = pd.DataFrame(test_features_list)
y_test = df_test['Label']

In [ ]:
# Predict
y_test_proba = rf.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_proba > OPTIMAL_THRESHOLD).astype(int)

# Classification Report
print("CLASSIFICATION REPORT:")
print(classification_report(y_test, y_test_pred,
                           target_names=['Legitimate', 'Phishing'],
                           digits=4))

# Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)
tn, fp, fn, tp = cm.ravel()

print("\nCONFUSION MATRIX:")
print(f"True Negative: {tn}")
print(f"False Positive: {fp} (Legitimate bị phân loại nhầm là Phishing)")
print(f"False Negative: {fn} (Phishing bị bỏ sót)")
print(f"True Positive: {tp}")

# Metrics
total = tn + fp + fn + tp
accuracy = (tp + tn) / total
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
auc = roc_auc_score(y_test, y_test_proba)

print(f"\nMETRICS:")
print(f"AUC Score: {auc:.4f}")
print(f"Accuracy: {accuracy*100:.2f}%")

# Error Analysis
fn_rate = fn / (fn + tp) if (fn + tp) > 0 else 0
fp_rate = fp / (fp + tn) if (fp + tn) > 0 else 0

print(f"\nERROR ANALYSIS:")
print(f"  False Negative Rate: {fn_rate*100:.2f}%")
print(f"  False Positive Rate: {fp_rate*100:.2f}%")

In [ ]:
# 1. Feature Importance Plot
plt.figure(figsize=(12, 8))
top_features = feature_importance.head(15)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance', fontsize=12)
plt.title('Top 15 Important Features', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# 2. Precision-Recall Curve
plt.figure(figsize=(10, 6))
plt.plot(recalls, precisions, linewidth=2, label='PR Curve')
plt.scatter(recalls[best_idx], precisions[best_idx], color='red', s=100, 
           marker='o', label=f'F2-optimal (t={OPTIMAL_THRESHOLD:.3f})', zorder=5)
plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curve', fontsize=14, fontweight='bold')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 3. Confusion Matrix Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', 
           xticklabels=['Legitimate', 'Phishing'],
           yticklabels=['Legitimate', 'Phishing'])
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 4. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_test_proba)

plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, linewidth=2, label=f'RF (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve', fontsize=14, fontweight='bold')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import pickle
import json
import inspect

SAVE_DIR = '/kaggle/working/artifacts_rf'
os.makedirs(SAVE_DIR, exist_ok=True)

# 1. Save model
model_path = os.path.join(SAVE_DIR, 'random_forest_model.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(rf, f, protocol=pickle.HIGHEST_PROTOCOL)
print(f"Model: {model_path}")

# 2. Save feature names
feature_names_path = os.path.join(SAVE_DIR, 'feature_names.pkl')
with open(feature_names_path, 'wb') as f:
    pickle.dump(list(X.columns), f)
print(f"Feature names: {feature_names_path}")

# 3. Save feature importance
importance_path = os.path.join(SAVE_DIR, 'feature_importance.csv')
feature_importance.to_csv(importance_path, index=False)
print(f"Feature importance: {importance_path}")

# 4. Save configuration
config = {
    'threshold': float(OPTIMAL_THRESHOLD),
    'hyperparameters': {
        'n_estimators': N_ESTIMATORS,
        'max_depth': MAX_DEPTH,
        'min_samples_split': MIN_SAMPLES_SPLIT,
        'min_samples_leaf': MIN_SAMPLES_LEAF,
        'max_features': MAX_FEATURES
    },
    'performance': {
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
        'auc': float(auc),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    },
    'data': {
        'total': len(df_train),
        'train_size': len(X_train),
        'val_size': len(X_val),
        'test_size': len(X_test)
    },
    'num_features': X.shape[1],
    'seed': SEED,
}

config_path = os.path.join(SAVE_DIR, 'config.json')
with open(config_path, 'w') as f:
    json.dump(config, f, indent=4)
print(f"Config: {config_path}")

# 5. Save feature extraction code
feature_code = inspect.getsource(extract_features)
feature_code_path = os.path.join(SAVE_DIR, 'feature_extraction.py')
with open(feature_code_path, 'w') as f:
    f.write('''import re
import numpy as np
from urllib.parse import urlparse
from collections import Counter

''')
    f.write(feature_code)
print(f"Feature extraction: {feature_code_path}")

# 6. Save deployment script
deployment_script = '''#!/usr/bin/env python3
"""
Phishing URL Detection - Random Forest Deployment
"""

import pickle
import json
import re
import pandas as pd
from feature_extraction import extract_features

# Load artifacts
with open('random_forest_model.pkl', 'rb') as f:
    model = pickle.load(f)

with open('config.json', 'r') as f:
    config = json.load(f)

with open('feature_names.pkl', 'rb') as f:
    feature_names = pickle.load(f)

threshold = config['threshold']

def clean_url(url):
    """Clean URL"""
    url = str(url).lower()
    url = re.sub(r'^https?://', '', url)
    url = re.sub(r'^www\\.', '', url)
    return url

def predict_url(url):
    """
    Predict if URL is phishing
    
    Args:
        url: URL string
    
    Returns:
        dict: Prediction results
    """
    # Clean
    cleaned_url = clean_url(url)
    
    # Extract features
    features = extract_features(cleaned_url)
    features_df = pd.DataFrame([features])[feature_names]
    
    # Predict
    probability = model.predict_proba(features_df)[0][1]
    is_phishing = probability > threshold
    
    return {
        'url': url,
        'cleaned_url': cleaned_url,
        'probability': float(probability),
        'is_phishing': bool(is_phishing),
        'confidence': 'high' if abs(probability - threshold) > 0.3 else 
                     'medium' if abs(probability - threshold) > 0.15 else 'low',
        'threshold': threshold
    }

if __name__ == "__main__":
    # Test
    test_urls = [
        "https://www.google.com",
        "http://paypal-verify.suspicious.xyz/login",
        "secure-banking-update.tk/account"
    ]
    
    print("Testing predictions:\\n")
    for url in test_urls:
        result = predict_url(url)
        print(f"URL: {result['url']}")
        print(f"  Phishing: {result['is_phishing']}")
        print(f"  Probability: {result['probability']:.4f}")
        print(f"  Confidence: {result['confidence']}")
        print()
'''

deployment_path = os.path.join(SAVE_DIR, 'predict.py')
with open(deployment_path, 'w') as f:
    f.write(deployment_script)
print(f"Deployment script: {deployment_path}")

# List artifacts
print("\nAll artifacts:")
for filename in sorted(os.listdir(SAVE_DIR)):
    filepath = os.path.join(SAVE_DIR, filename)
    size = os.path.getsize(filepath)
    size_str = f"{size/1024/1024:.1f} MB" if size > 1024*1024 else f"{size/1024:.1f} KB"
    print(f"  - {filename} ({size_str})")